# BIPN 162 Final Project
ByEmily McClung, Naomi Ortega, and Isabel Rodriguez

In [ ]:
!pip install pynwb h5py torch scikit-learn configmypy tensorboardX

In [ ]:
#CLone the DUNL algorithm GitHub Repository
!git clone https://github.com/btolooshams/dunl-compneuro.git

In [2]:
import numpy as np
import torch
import sklearn
import h5py
import configmypy

#Import the DUNL code
import sys
sys.path.append("dunl-compneuro")
import dunl

#Imports for data
import os
from pynwb import NWBHDF5IO
os.makedirs("data", exist_ok=True)

In [ ]:
!ls dunl-compneuro/dunl

## Applying DUNL to another dataset

We will now apply the DUNL analysis method to a dataset of Fiber Photometry and Optogenetics recordings.

In [ ]:
#download the dataset DANDI 000971
!pip install dandi
!/home/emcclung/.local/bin/dandi download \
"https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-028-392" \
--output-dir data

In [11]:
#look through the files to open up a specific one
os.listdir("data")

#create list  of file names
data_dir = "data"
files = os.listdir(data_dir)

#pick a NWB file
actual_filename = [f for f in files if f.endswith('.nwb')][0]
file_path = os.path.join(data_dir, actual_filename)

#open the file
with NWBHDF5IO(file_path, "r") as io:
    nwb = io.read()
    print(nwb)

root pynwb.file.NWBFile at 0x140567923424464
Fields:
  experiment_description: Compulsive behavior is a defining feature of disorders such as substance use disorders. Current evidence suggests that corticostriatal circuits control the expression of established compulsions, but little is known about the mechanisms regulating the development of compulsions. We hypothesized that dopamine, a critical modulator of striatal synaptic plasticity, could control alterations in corticostriatal circuits leading to the development of compulsions (defined here as continued reward seeking in the face of punishment). We used dual-site fiber photometry to measure dopamine axon activity in the dorsomedial striatum (DMS) and the dorsolateral striatum (DLS) as compulsions emerged. Individual variability in the speed with which compulsions emerged was predicted by DMS dopamine axon activity. Amplifying this dopamine signal accelerated animals' transitions to compulsion, whereas inhibition delayed it. In co

/home/n2ortega/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-fiber-photometry': 
 ndx-fiber-photometry defines OpticalFiber.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines ExcitationSource.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines Photodetector.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines DichroicMirror.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines BandOpticalFilter.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines EdgeOpticalFilter.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel. 
T

In [13]:
#path to one subjects data
file_path = "data/sub-028-392/sub-028-392_ses-FP-PR-2020-07-09T13-01-26.nwb"

with NWBHDF5IO(file_path, "r") as io:
    nwb = io.read()
    
    #get the photometry signal
    fp_series = nwb.acquisition['fiber_photometry_response_series']
    raw_signal = fp_series.data[:]
    rate = fp_series.rate
    start = fp_series.starting_time
    timestamps = start + (np.arange(len(raw_signal)) / rate)
    
    signal = raw_signal[:, 0]

    #get the behavioral events
    beh = nwb.processing['behavior'].data_interfaces
    
    #get timestamps for each event type
    reward_times = beh['right_reward_times'].timestamps[:]
    left_pokes = beh['left_nose_poke_times'].timestamps[:]
    right_pokes = beh['right_nose_poke_times'].timestamps[:]
 
    #combine nose pokes into into one event
    all_pokes = np.sort(np.concatenate([left_pokes, right_pokes]))

#format and save
data_dict = {
    "signal": signal.astype(np.float32),
    "timestamps": timestamps.astype(np.float32),
    "events": {
        "reward": reward_times.astype(np.float32),
        "nose_poke": all_pokes.astype(np.float32)
    },
    "metadata": {
        "sampling_rate": rate,
        "session_id": "sub-028-392_2020-07-09"
    }
}

np.save("data_for_dunl.npy", data_dict)

print("finish preprocess")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'data/sub-028-392/sub-028-392_ses-FP-PR-2020-07-09T13-01-26.nwb', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [14]:
#load the data we extracted earlier
data_path = "dunl-compneuro/data/data_for_dunl.npy"
data = np.load(data_path, allow_pickle=True).item()

bin_size = 0.02  
kernel_length_sec = 5.0  
bins_per_kernel = int(kernel_length_sec / bin_size)

# organize time stamps
max_time = data['timestamps'][-1]
time_bins = np.arange(0, max_time, bin_size)

#bin the photometry signal
binned_signal, _ = np.histogram(data['timestamps'], bins=time_bins, weights=data['signal'])
counts, _ = np.histogram(data['timestamps'], bins=time_bins)

final_signal = np.divide(binned_signal, counts, out=np.zeros_like(binned_signal), where=counts!=0)

#normalization
final_signal = (final_signal - np.mean(final_signal)) / np.std(final_signal)

#bin events 
reward_vec, _ = np.histogram(data['events']['reward'], bins=time_bins)
poke_vec, _ = np.histogram(data['events']['nose_poke'], bins=time_bins)

#save for the DUNL Model
dunl_input = {
    "signal": torch.tensor(final_signal).float(),
    "reward": torch.tensor(reward_vec).float().clamp(0, 1), 
    "nose_poke": torch.tensor(poke_vec).float().clamp(0, 1)
}

#create a folder for the result
os.makedirs("dunl_ready_data", exist_ok=True)
torch.save(dunl_input, "dunl_ready_data/session_01.pt")

print("Preprocessing Complete")

FileNotFoundError: [Errno 2] No such file or directory: 'dunl-compneuro/data/data_for_dunl.npy'

## 4. Conclusion & Discussion (~400-500 words)

Discussion of your results and how they address your scientific question.
Limitations of analysis discussed.
What additional experiments would be interesting, and what data would you need?

## 5. Reflection  (~100-200 words)

In 100-200 words, please reflect on this project as a whole, from the proposal to the final product. How did you integrate and address feedback on the proposal? What was difficult? What was rewarding? Did this go as planned?

## 6. List of Contributions & AI Usage Statement

7. Code Clarity, Readability & Style